# Week 3 Day 3 — Tokenizers

**Local Mac reference notebook.** Same lab as the course Colab, rewritten to run here: Hugging Face login comes from `.env`, and there is no GPU check. Tokenizers run on CPU. Loading a tokenizer does **not** load model weights.

## What this notebook is for

An LLM never sees Python strings or `messages` lists. It sees a sequence of integers (**token IDs**). A tokenizer is the bidirectional map:

```
text  →  encode()  →  token IDs  →  model
text  ←  decode()  ←  token IDs  ←  model output
```

That map is model-specific. Llama, Phi, DeepSeek, and Qwen each have their own vocabulary and their own chat template. Mixing them (encoding with one tokenizer, generating with another model) produces garbage.

| Idea | Why it matters later |
| --- | --- |
| Token count ≠ word count | Context windows, API bills, and RAG chunk sizes are in **tokens** |
| `apply_chat_template` | Instruct models expect a tagged prompt, not a raw question |
| Special tokens (BOS, EOS, role tags) | They are in the ID sequence even when you did not type them |
| Vocab is frozen with the model | You cannot invent IDs the model was not trained on |

Day 2 used `pipeline()`, which hid the tokenizer. This notebook is the layer underneath.

## How to use this as a cheat sheet

1. Run **Setup** once per kernel (imports + Hugging Face login).
2. Jump to the section you need. Later cells assume Setup has run; they do not assume you ran every demo.
3. Copy patterns: `AutoTokenizer.from_pretrained(id)`, `encode` / `decode`, `apply_chat_template`.
4. Keep tokenizer and model IDs in lockstep. If you change the model, change the tokenizer.

Course Colab (original): [Week 3 Day 3 tokenizers](https://colab.research.google.com/drive/1WD6Y2N7ctQi1X9wa6rpkg8UfyA4iSVuz?usp=sharing)


## Setup

Packages live in this project's `.venv`. Skip `%pip` / `!pip` here.

Local differences from the Colab notebook:

- No `google.colab.userdata` — load `HF_TOKEN` from `.env`
- No `nvidia-smi` / T4 check — tokenizers do not use a GPU
- No `trust_remote_code=True` — these checkpoints ship a standard tokenizer; do not execute Hub Python unless a model actually requires it

First `from_pretrained` for each model ID downloads tokenizer files into the Hugging Face cache (small). It does not download the full LLM.


In [ ]:
# Confirm the packages this notebook needs. Re-run Setup after a kernel restart.
import transformers

print(f"transformers  {transformers.__version__}")


In [ ]:
import os

from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer


### Hugging Face login

Needed for **gated** models (Llama) and to avoid anonymous Hub rate limits. Create a token at https://huggingface.co/settings/tokens (write scope if you will push models later) and put `HF_TOKEN=hf_...` in the project `.env`.

Llama 3.1 also requires accepting Meta's terms on the model card: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B  
Use the same email as your Hugging Face account. Approval is usually quick and covers the 3.1 family (base and Instruct).

If `from_pretrained` fails after login:

1. Token starts with `hf_` and has at least read access
2. The model page shows you have access
3. Re-run `login` after rotating a token


In [ ]:
load_dotenv(override=True)
hf_token = os.getenv("HF_TOKEN")

# Tokens always start with hf_. login() returns None — do not assign it to hf_token.
if hf_token and hf_token.startswith("hf_"):
    print("HF key looks good so far")
else:
    raise ValueError("HF_TOKEN not found. Add HF_TOKEN=hf_... to your .env file.")

login(hf_token, add_to_git_credential=True)


## 1. Encode, decode, and vocabulary (Llama 3.1 base)

`AutoTokenizer.from_pretrained(model_id)` loads the tokenizer that was trained with that checkpoint. Same Hub ID you would use for the model, but only tokenizer JSON / vocab files come down.

Core API:

| Call | Returns |
| --- | --- |
| `tokenizer.encode(text)` | `list[int]` token IDs |
| `tokenizer.decode(ids)` | reconstructed string |
| `tokenizer.convert_ids_to_tokens(ids)` | vocab piece strings (one per ID) |
| `tokenizer.apply_chat_template(messages)` | Instruct prompt string or IDs (section 2) |

`encode(..., add_special_tokens=True)` is the default. For Llama that prepends BOS (`<|begin_of_text|>`, ID 128000). That extra ID is why `len(tokens)` is often one more than the pieces you would guess from the words alone.


In [ ]:
# Base (non-Instruct) Llama 3.1 — pretrained LM tokenizer, not the chat template variant.
# Keep this object; later cells load Instruct into a different variable instead of overwriting.
llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B")
print(llama_tokenizer.__class__.__name__)
print(llama_tokenizer.name_or_path)


In [ ]:
text = "I am excited to show Tokenizers in action to my LLM engineers"

# encode: string → list of integers the model would actually consume.
tokens = llama_tokenizer.encode(text)
print(tokens)

character_count = len(text)
word_count = len(text.split(" "))  # whitespace split, not linguistic words
token_count = len(tokens)

print(
    f"There are {character_count} characters, {word_count} words and {token_count} tokens"
)


In [ ]:
# Round-trip. decode() default skip_special_tokens=False, so BOS may appear in the string.
print("decode(tokens):")
print(repr(llama_tokenizer.decode(tokens)))
print()
print("decode(tokens, skip_special_tokens=True):")
print(repr(llama_tokenizer.decode(tokens, skip_special_tokens=True)))
print()

# convert_ids_to_tokens: the actual vocab pieces (spaces often show as a leading Ġ or a leading space).
print("convert_ids_to_tokens:")
print(llama_tokenizer.convert_ids_to_tokens(tokens))
print()

# Course trick: batch_decode on a 1D list of ints treats each ID as its own sequence,
# so you get one decoded string per token. convert_ids_to_tokens is the clearer API.
print("batch_decode(tokens)  [one string per ID]:")
print(llama_tokenizer.batch_decode(tokens))


In [ ]:
# Same sentence with and without the automatic BOS/EOS wrapping.
with_special = llama_tokenizer.encode(text, add_special_tokens=True)
without_special = llama_tokenizer.encode(text, add_special_tokens=False)

print(f"add_special_tokens=True:  {len(with_special)} IDs → {with_special[:4]} ...")
print(f"add_special_tokens=False: {len(without_special)} IDs → {without_special[:4]} ...")
print()
print("bos_token / bos_token_id:", llama_tokenizer.bos_token, llama_tokenizer.bos_token_id)
print("eos_token / eos_token_id:", llama_tokenizer.eos_token, llama_tokenizer.eos_token_id)
print("pad_token / pad_token_id:", llama_tokenizer.pad_token, llama_tokenizer.pad_token_id)
print()
print("Rule of thumb: count tokens the same way the model will see them.")
print("For a raw completion, that usually means add_special_tokens=True.")
print("For chat, count the IDs from apply_chat_template(..., tokenize=True) instead.")


In [ ]:
# vocab_size is the embedding table width the model was trained with.
# len(vocab) is the Python dict of pieces; it can differ slightly from vocab_size.
print("vocab_size:        ", llama_tokenizer.vocab_size)
print("len(tokenizer.vocab):", len(llama_tokenizer.vocab))
print()

# Pieces added on top of the base BPE / tiktoken vocab (role tags, tool tokens, etc.).
# Do not print tokenizer.vocab — it is 100k+ entries.
print("get_added_vocab():")
print(llama_tokenizer.get_added_vocab())


### Section 1 summary

- **encode** produces the ID sequence; **decode** rebuilds text; **convert_ids_to_tokens** shows the pieces.
- Token count is its own unit. Subwords (`Token` + `izers`), punctuation, and special tokens all add IDs.
- Defaults matter: `add_special_tokens=True` and `skip_special_tokens=False` change both length and the decoded string.
- Inspect `bos_token_id` / `eos_token_id` / `pad_token_id` before you pad a batch. Llama 3.1 base often has `pad_token is None` until you set one.


## 2. Instruct models and `apply_chat_template`

Many Hub models have an **Instruct** (or **Chat**) variant. Same tokenizer family, extra training so the model expects a prompt with system / user / assistant roles.

You still pass the familiar OpenAI-style list of dicts in Python. The tokenizer turns that list into **one string** (or one ID sequence) using a Jinja **chat template** stored on the tokenizer.

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"},
]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,            # return a string; True returns token IDs
    add_generation_prompt=True,  # append the assistant header so the model fills in a reply
)
```

`add_generation_prompt=True` is what you want for inference. Leave it `False` when the last message is already an assistant turn (for example training or scoring a completed conversation).

Base Llama 3.1 may have a weaker or missing chat template. Use the **Instruct** tokenizer (and Instruct weights) for chat.


In [ ]:
llama_instruct_tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct"
)

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"},
]

# tokenize=False: inspect the tagged string. tokenize=True: the IDs you would feed the model.
prompt = llama_instruct_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print(prompt)
print("---")
prompt_ids = llama_instruct_tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True
)
print(f"chat template token count: {len(prompt_ids)}")


### Crucial "aha" moment

Weeks 1–2 made it look like models consume a list of Python dicts:

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"},
]
```

An LLM is a statistical model over **integer sequences**. It cannot take dicts. The missing conversion is:

1. `apply_chat_template` turns `messages` into a tagged string (system / user / assistant delimiters).
2. The tokenizer splits that string into pieces (tokens).
3. Each piece is replaced with a token ID.

> The input to an LLM is a sequence of token IDs. The output is a probability distribution over the next token ID.

Hosted APIs (OpenAI, Anthropic, Gemini) do this on their side. Hugging Face Instruct models make you (or `pipeline` / `apply_chat_template`) do it explicitly.


### Section 2 summary

- **Instruct** = same tokenizer family + a chat template + weights trained to follow that template.
- Always use `apply_chat_template` (or a pipeline that calls it). Hand-rolling `<|user|>` tags is how prompts silently mismatch.
- `tokenize=False` is for reading the prompt; `tokenize=True` is the budget you should log.
- `add_generation_prompt=True` when you want the model to reply.


## 3. Same text, different tokenizers

Three more Instruct tokenizers. The sentence is identical; the ID lists are not. Chat templates differ too (Llama headers vs Phi / ChatML-style tags vs DeepSeek's format).

Still tokenizer-only: no model weights, no GPU.

| Variable | Hub ID | Why it is in the lab |
| --- | --- | --- |
| `llama_instruct_tokenizer` | `meta-llama/Meta-Llama-3.1-8B-Instruct` | Chat template baseline |
| `phi4_tokenizer` | `microsoft/Phi-4-mini-instruct` | Different vocab + different tags |
| `deepseek_tokenizer` | `deepseek-ai/DeepSeek-V3.1` | Another family; huge model, tiny tokenizer download |
| `qwen_tokenizer` | `Qwen/Qwen2.5-Coder-7B-Instruct` | Code-aware pieces |


In [ ]:
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"


In [ ]:
phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

text = (
    "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"
)

print("Llama 3.1 Instruct:")
llama_ids = llama_instruct_tokenizer.encode(text)
print(llama_ids)
print(llama_instruct_tokenizer.convert_ids_to_tokens(llama_ids))
print(f"{len(llama_ids)} tokens")

print("\nPhi-4 mini Instruct:")
phi_ids = phi4_tokenizer.encode(text)
print(phi_ids)
print(phi4_tokenizer.convert_ids_to_tokens(phi_ids))
print(f"{len(phi_ids)} tokens")
print()
print("IDs are not interchangeable. The integer 40 is a different piece in each vocab.")


In [ ]:
print("Llama 3.1 Instruct chat template:")
print(
    llama_instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
)
print("\nPhi-4 mini Instruct chat template:")
print(
    phi4_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
)


In [ ]:
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)

print("Llama 3.1 Instruct IDs:")
print(llama_instruct_tokenizer.encode(text))
print(f"{len(llama_instruct_tokenizer.encode(text))} tokens")

print("\nPhi-4 mini Instruct IDs:")
print(phi4_tokenizer.encode(text))
print(f"{len(phi4_tokenizer.encode(text))} tokens")

print("\nDeepSeek-V3.1 IDs:")
print(deepseek_tokenizer.encode(text))
print(f"{len(deepseek_tokenizer.encode(text))} tokens")


In [ ]:
print("Llama:")
print(
    llama_instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
)
print("\nPhi-4:")
print(
    phi4_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
)
print("\nDeepSeek:")
print(
    deepseek_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
)


In [ ]:
# Code models often keep identifiers and keywords as coarser pieces than a general tokenizer.
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)

code = """
def hello_world(person):
  print("Hello", person)
"""

code_ids = qwen_tokenizer.encode(code)
print(f"Qwen2.5-Coder: {len(code_ids)} tokens\n")
for token_id, piece in zip(code_ids, qwen_tokenizer.convert_ids_to_tokens(code_ids)):
    # decode([id]) applies the tokenizer's decoding rules; piece is the raw vocab string.
    print(f"{token_id:>8}  {piece!r:20}  {qwen_tokenizer.decode([token_id])!r}")


In [ ]:
def inspect_tokenization(tokenizer, text, *, add_special_tokens=True):
    """Print ID → piece for any Hugging Face tokenizer. Copy this into later notebooks."""
    ids = tokenizer.encode(text, add_special_tokens=add_special_tokens)
    pieces = tokenizer.convert_ids_to_tokens(ids)
    print(f"{tokenizer.name_or_path}: {len(ids)} tokens")
    for token_id, piece in zip(ids, pieces):
        print(f"  {token_id:>8}  {piece!r}")
    return ids


def count_chat_tokens(tokenizer, messages, *, add_generation_prompt=True):
    """Context-window / cost counter for Instruct prompts."""
    ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=add_generation_prompt
    )
    return len(ids)


# Smoke-check the helpers on the sentence already in `text`.
inspect_tokenization(llama_instruct_tokenizer, text, add_special_tokens=False)
print()
print("chat tokens (Llama Instruct):", count_chat_tokens(llama_instruct_tokenizer, messages))


### Section 3 summary

- Identical text → different ID lists and different lengths. Never reuse IDs across families.
- Identical `messages` → different chat strings. Use **that model's** `apply_chat_template`.
- A 671B-class checkpoint (DeepSeek-V3.1) still has a small tokenizer. `AutoTokenizer.from_pretrained` is safe; `AutoModel.from_pretrained` is not, on this machine.
- Coder tokenizers are still BPE/unigram; they are just trained so code tokens fragment less awkwardly. Always inspect with `convert_ids_to_tokens`.


## Conclusion

A tokenizer is the contract between text and the integer sequence a model was trained on. Day 3 is that contract, not a new kind of model.

**Takeaways to reuse later**

1. **Load the tokenizer from the same Hub ID as the model** (or the official tokenizer ID the card names).
2. **Count tokens, not words.** Context limits, batching, and billed usage are token IDs.
3. **`encode` / `decode` for raw text; `apply_chat_template` for Instruct chat.**
4. **`add_generation_prompt=True` for inference** when the model should produce an assistant turn.
5. **Special tokens are real IDs.** Check `add_special_tokens` and `skip_special_tokens` before you compare lengths.
6. **Chat templates are not portable.** Llama tags ≠ Phi tags ≠ DeepSeek tags.
7. **Tokenizer download ≠ model download.** This notebook stays on CPU on purpose.

**Copy-paste surface**

```python
tokenizer = AutoTokenizer.from_pretrained(model_id)
ids = tokenizer.encode(text)                    # or add_special_tokens=False
text_out = tokenizer.decode(ids, skip_special_tokens=True)
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
n = len(tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True
))
```

**When this notebook is the wrong tool**

- Running generation — you need the model (Day 4) or `pipeline("text-generation")` (Day 2)
- Training / fine-tuning — padding, truncation, and a collator (later weeks)
- Hosted APIs — they tokenize server-side; you still care about token counts for cost

**Local Mac checklist if something breaks**

- Kernel restart → re-run Setup (`import` + `login`)
- `userdata` / `nvidia-smi` / `!pip` → Colab-only; this notebook already avoids them
- `401` / `gated repo` on Llama → `HF_TOKEN` in `.env` and Meta terms accepted on the model card
- Chat looks "off" → you used the base tokenizer, or `add_generation_prompt=False`, or another family's template
